<a href="https://colab.research.google.com/github/nachocabra/tesis-biomecanica-caballos/blob/main/tesis_pipeline_pose_estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline de pose estimation — Tesis
Extracción de métricas biomecánicas de caballos de carrera a partir de
video de transmisión pública, mediante el modelo preentrenado
SuperAnimal-Quadruped (DeepLabCut).

## 1. Instalación de DeepLabCut

Instalamos la librería DeepLabCut, incluyendo el extra `modelzoo`, que da
acceso a los modelos preentrenados (entre ellos, SuperAnimal-Quadruped,
el que usamos en este trabajo) sin necesidad de entrenar uno propio.

In [ ]:
!pip install deeplabcut[gui,modelzoo]

## 2. Conexión con Google Drive

Montamos Google Drive dentro de la sesión de Colab, para que los videos
y los resultados generados queden guardados de forma permanente, en
lugar de perderse cuando se corte la sesión.

## 3. Estructura de carpetas y rutas de trabajo

Definimos la organización de carpetas dentro de Drive:

- `videos_originales/`: los videos completos, descargados con yt-dlp.
- `clips/`: recortes cortos de tramos específicos, usados para las pruebas.
- `resultados/`: salidas del modelo (coordenadas, video con keypoints
  dibujados), separadas en una subcarpeta por cada prueba realizada.

También definimos variables con las rutas a los videos originales, para
no repetir las rutas completas en cada celda.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/Tesis/videos', exist_ok=True)

In [ ]:
import os
os.listdir('/content/drive/MyDrive/Tesis/videos')

['Gulfstream Park September 20, 2026 Race 9.mp4',
 '2026 Jockey Club Gold Cup- Forever Young.webm']

---
# Pruebas exploratorias de pose estimation

Primeras corridas del modelo sobre clips cortos, para verificar
viabilidad técnica antes de definir el pipeline completo.

In [ ]:
video_gulfstream = '/content/drive/MyDrive/Tesis/videos/Gulfstream Park September 20, 2026 Race 9.mp4'
video_belmont = '/content/drive/MyDrive/Tesis/videos/2026 Jockey Club Gold Cup- Forever Young.webm'

## 4. Prueba 1: recorte de Gulfstream Park

Cortamos un clip de 10 segundos del video de Gulfstream Park, a partir
del minuto 1:49, correspondiente al tramo identificado como lateral
limpio (sin obstrucción de la valla), según el relevamiento manual de
ángulos de cámara hecho previamente.

In [ ]:
!ffmpeg -i "{video_gulfstream}" -ss 00:01:49 -t 10 -c copy /content/drive/MyDrive/Tesis/videos/gulfstream_clip.mp4


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

## 5. Prueba 1: primera corrida del modelo (SuperAnimal-Quadruped)

Corremos el modelo de pose estimation preentrenado sobre el clip de
Gulfstream, para verificar si el pipeline funciona técnicamente sobre
video de carrera real.

Nota: en esta corrida, el parámetro `video_adapt=True` (que reduce el
parpadeo/jitter de las predicciones mediante un reentrenamiento breve)
causó un error de memoria de GPU (`CUDA out of memory`) en la etapa de
ajuste. El análisis principal, sin embargo, se completó correctamente
antes de ese punto, generando resultados válidos.

In [ ]:
import deeplabcut

clip_path = '/content/drive/MyDrive/Tesis/videos/gulfstream_clip.mp4'

deeplabcut.video_inference_superanimal(
    [clip_path],
    "superanimal_quadruped",
    model_name="hrnet_w32",
    detector_name="fasterrcnn_resnet50_fpn_v2",
    scale_list=range(200, 1000, 100),
    video_adapt=True
)

Running video inference on ['/content/drive/MyDrive/Tesis/videos/gulfstream_clip.mp4'] with superanimal_quadruped_hrnet_w32
Using pytorch for model hrnet_w32
Loading.... superanimal_quadruped_hrnet_w32


/usr/local/lib/python3.13/dist-packages/deeplabcut/pose_estimation_pytorch/config/data.py:139: UserWarning: Be careful! Do not train pose models with horizontal flips if you have symmetric keypoints!
  build_transforms(v)


superanimal_quadruped_hrnet_w32.pt: reconstructing file:   0%|          |  0.00B /  118MB            

superanimal_quadruped_hrnet_w32.pt: downloading bytes:           |  0.00B            

Loading.... superanimal_quadruped_fasterrcnn_resnet50_fpn_v2


...

Saving results to /content/drive/MyDrive/Tesis/videos
Saving results in /content/drive/MyDrive/Tesis/videos/gulfstream_clip_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2.h5 and /content/drive/MyDrive/Tesis/videos/gulfstream_clip_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_full.pickle
Duration of video [s]: 9.74, recorded with 29.97 fps!
Overall # of frames: 292 with cropped frame dimensions: 1920 1080
Generating frames and creating video.


100%|██████████| 271/271 [00:11<00:00, 23.83it/s]


Video with predictions was saved as /content/drive/MyDrive/Tesis/videos/gulfstream_clip_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_labeled_before_adapt.mp4
Video frames being extracted to /content/drive/MyDrive/Tesis/videos/pseudo_gulfstream_clip/images for video adaptation.
Constructing pseudo dataset at /content/drive/MyDrive/Tesis/videos/pseudo_gulfstream_clip
Truncated both arrays to length 271
Running video adaptation with following parameters:
  video adaptation batch size: 8
  (pose training) pose_epochs: 4
  (pose) save_epochs: 1
  detector_epochs: 4
  detector_save_epochs: 1



/usr/local/lib/python3.13/dist-packages/deeplabcut/pose_estimation_pytorch/config/data.py:139: UserWarning: Be careful! Do not train pose models with horizontal flips if you have symmetric keypoints!
  build_transforms(v)
/usr/local/lib/python3.13/dist-packages/deeplabcut/pose_estimation_pytorch/apis/training.py:137: UserWarning: Be careful! Do not train pose models with horizontal flips if you have symmetric keypoints!
  transform = build_transforms(run_config["data"]["train"])
Data Transforms:
  Training:   Compose([
  HorizontalFlip(always_apply=False, p=0.5),
  Affine(always_apply=False, p=0.5, interpolation=1, mask_interpolation=0, cval=0, mode=0, scale={'x': (1.0, 1.0), 'y': (1.0, 1.0)}, translate_percent=None, translate_px={'x': (-40, 40), 'y': (-40, 40)}, rotate=(-30, 30), fit_output=False, shear={'x': (0.0, 0.0), 'y': (0.0, 0.0)}, cval_mask=0, keep_ratio=True, rotate_method='largest_box'),
  Normalize(always_apply=False, p=1.0, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.

OutOfMemoryError: CUDA out of memory. Tried to allocate 126.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 121.81 MiB is free. Including non-PyTorch memory, this process has 14.44 GiB memory in use. Of the allocated memory 14.13 GiB is allocated by PyTorch, and 189.88 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 6. Reorganización de carpetas

Reorganizamos los archivos generados hasta este punto (que habían
quedado todos en una única carpeta) en la estructura definitiva descrita
en el paso 3, y eliminamos archivos intermedios sin uso (generados por
el intento fallido de ajuste del paso anterior).

In [ ]:
import os
import shutil

base = '/content/drive/MyDrive/Tesis'

# Creamos las carpetas nuevas
os.makedirs(f'{base}/videos_originales', exist_ok=True)
os.makedirs(f'{base}/clips', exist_ok=True)
os.makedirs(f'{base}/resultados', exist_ok=True)

# Movemos los videos originales
shutil.move(f'{base}/videos/2026 Jockey Club Gold Cup- Forever Young.webm', f'{base}/videos_originales/')
shutil.move(f'{base}/videos/Gulfstream Park September 20, 2026 Race 9.mp4', f'{base}/videos_originales/')

# Movemos el clip
shutil.move(f'{base}/videos/gulfstream_clip.mp4', f'{base}/clips/')

# Movemos los resultados del modelo (todo lo que empieza con "gulfstream_clip_superanimal")
for archivo in os.listdir(f'{base}/videos'):
    if archivo.startswith('gulfstream_clip_superanimal'):
        shutil.move(f'{base}/videos/{archivo}', f'{base}/resultados/')

# Borramos la carpeta basura del intento fallido
shutil.rmtree(f'{base}/videos/pseudo_gulfstream_clip')

# Confirmamos que la carpeta "videos" vieja quedó vacía, y la borramos
os.rmdir(f'{base}/videos')

Error: Destination path '/content/drive/MyDrive/Tesis/videos_originales/2026 Jockey Club Gold Cup- Forever Young.webm' already exists

## 7. Actualización de rutas

Actualizamos las variables de rutas para que apunten a la nueva
estructura de carpetas.

In [ ]:
print("En videos_originales:", os.listdir(f'{base}/videos_originales'))
print("En videos:", os.listdir(f'{base}/videos'))

En videos_originales: ['Gulfstream Park September 20, 2026 Race 9.mp4', '2026 Jockey Club Gold Cup- Forever Young.webm']


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Tesis/videos'

In [ ]:
print("Tesis (raíz):", os.listdir(base))
print()
print("clips:", os.listdir(f'{base}/clips'))
print()
print("resultados:", os.listdir(f'{base}/resultados'))

Tesis (raíz): ['videos_originales', 'clips', 'resultados']

clips: ['gulfstream_clip.mp4', 'belmont_clip.mp4', 'belmont_clip_h264.mp4']

resultados: ['gulfstream_clip_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2.h5', 'gulfstream_clip_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_before_adapt.json', 'gulfstream_clip_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_labeled_before_adapt.mp4']


In [ ]:
video_gulfstream = f'{base}/videos_originales/Gulfstream Park September 20, 2026 Race 9.mp4'
video_belmont = f'{base}/videos_originales/2026 Jockey Club Gold Cup- Forever Young.webm'
clip_path = f'{base}/clips/gulfstream_clip.mp4'

In [ ]:
os.listdir(f'{base}/videos_originales')

['Gulfstream Park September 20, 2026 Race 9.mp4',
 '2026 Jockey Club Gold Cup- Forever Young.webm']

## 8. Prueba 2: recorte de Belmont Park

Cortamos un clip de 6 segundos del video de Belmont Park (transmitido
por ESPN), a partir del minuto 2:08, correspondiente al tramo lateral
limpio identificado en el relevamiento manual.

In [ ]:
!ffmpeg -i "{video_belmont}" -ss 00:02:08 -t 6 -c copy {base}/clips/belmont_clip.mp4

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

## 9. Prueba 2: primer intento fallido — incompatibilidad de códec

Al intentar correr el modelo sobre este clip, el análisis procesó 0 de
176 cuadros. La causa identificada fue que el video de origen está
codificado en AV1, un códec que la herramienta de lectura de video usada
internamente por DeepLabCut no logra decodificar correctamente, a
diferencia de H.264 (usado en el clip de Gulfstream, que sí funcionó sin
problemas).

In [ ]:
deeplabcut.video_inference_superanimal(
    [f'{base}/clips/belmont_clip.mp4'],
    "superanimal_quadruped",
    model_name="hrnet_w32",
    detector_name="fasterrcnn_resnet50_fpn_v2",
    scale_list=range(200, 1000, 100),
    video_adapt=False
)

Found invalid empty/null/None value for `Task` in the project config. This is only supported for legacy compatibility. Treating `Task` as unset and using the field default instead.
Found invalid empty/null/None value for `scorer` in the project config. This is only supported for legacy compatibility. Treating `scorer` as unset and using the field default instead.
Found invalid empty/null/None value for `date` in the project config. This is only supported for legacy compatibility. Treating `date` as unset and using the field default instead.
Found invalid empty/null/None value for `multianimalproject` in the project config. This is only supported for legacy compatibility. Treating `multianimalproject` as unset and using the field default instead.
Found invalid empty/null/None value for `project_path` in the project config. This is only supported for legacy compatibility. Treating `project_path` as unset and using the field default instead.
Found invalid empty/null/None value for `video_

Running video inference on ['/content/drive/MyDrive/Tesis/clips/belmont_clip.mp4'] with superanimal_quadruped_hrnet_w32
Using pytorch for model hrnet_w32
Processing video /content/drive/MyDrive/Tesis/clips/belmont_clip.mp4
Starting to analyze /content/drive/MyDrive/Tesis/clips/belmont_clip.mp4
Video metadata: 
  Overall # of frames:    176
  Duration of video [s]:  5.87
  fps:                    29.97
  resolution:             w=1280, h=720

Running detector with batch size 1


  0%|          | 0/176 [00:00<?, ?it/s]


Running pose prediction with batch size 1


  0%|          | 0/176 [00:00<?, ?it/s]
The video metadata indicates that there 176 in the video, but only 0 were able to be processed. This can happen if the video is corrupted. You can try to fix the issue by re-encoding your video (tips on how to do that: https://deeplabcut.github.io/DeepLabCut/docs/recipes/io.html#tips-on-video-re-encoding-and-preprocessing)


RuntimeError: No pose predictions were made for video /content/drive/MyDrive/Tesis/clips/belmont_clip.mp4. Were no individuals detected?

## 10. Prueba 2: re-codificación a H.264

Convertimos el clip de Belmont del códec AV1 a H.264, para resolver la
incompatibilidad detectada en el paso anterior.


In [ ]:
!ffmpeg -i "{base}/clips/belmont_clip.mp4" -c:v libx264 -c:a aac {base}/clips/belmont_clip_h264.mp4

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

## 11. Prueba 2: corrida del modelo sobre el clip corregido

Corremos el modelo sobre el clip de Belmont ya convertido a H.264,
guardando los resultados en una subcarpeta separada de los de la Prueba 1,
para no mezclar los resultados de ambos hipódromos.

In [ ]:
!ffmpeg -i "{base}/clips/belmont_clip.mp4" -c:v libx264 -c:a aac {base}/clips/belmont_clip_h264.mp4

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [ ]:

import deeplabcut

In [ ]:
import os

os.makedirs(f'{base}/resultados/prueba2_belmont', exist_ok=True)

deeplabcut.video_inference_superanimal(
    [f'{base}/clips/belmont_clip_h264.mp4'],
    "superanimal_quadruped",
    model_name="hrnet_w32",
    detector_name="fasterrcnn_resnet50_fpn_v2",
    scale_list=range(200, 1000, 100),
    video_adapt=False,
    dest_folder=f'{base}/resultados/prueba2_belmont'
)

Running video inference on ['/content/drive/MyDrive/Tesis/clips/belmont_clip_h264.mp4'] with superanimal_quadruped_hrnet_w32
Using pytorch for model hrnet_w32


/usr/local/lib/python3.13/dist-packages/deeplabcut/pose_estimation_pytorch/config/data.py:139: UserWarning: Be careful! Do not train pose models with horizontal flips if you have symmetric keypoints!
  build_transforms(v)


Processing video /content/drive/MyDrive/Tesis/clips/belmont_clip_h264.mp4
Starting to analyze /content/drive/MyDrive/Tesis/clips/belmont_clip_h264.mp4
Video metadata: 
  Overall # of frames:    180
  Duration of video [s]:  6.01
  fps:                    29.97
  resolution:             w=1280, h=720

Running detector with batch size 1


100%|██████████| 180/180 [00:29<00:00,  6.04it/s]


Running pose prediction with batch size 1


100%|██████████| 180/180 [00:11<00:00, 15.86it/s]


Saving results to /content/drive/MyDrive/Tesis/resultados/prueba2_belmont
Saving results in /content/drive/MyDrive/Tesis/resultados/prueba2_belmont/belmont_clip_h264_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2.h5 and /content/drive/MyDrive/Tesis/resultados/prueba2_belmont/belmont_clip_h264_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_full.pickle
Duration of video [s]: 6.01, recorded with 29.97 fps!
Overall # of frames: 180 with cropped frame dimensions: 1280 720
Generating frames and creating video.


100%|██████████| 180/180 [00:03<00:00, 47.26it/s]

Video with predictions was saved as /content/drive/MyDrive/Tesis/resultados/prueba2_belmont/belmont_clip_h264_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_labeled_before_adapt.mp4


{'/content/drive/MyDrive/Tesis/clips/belmont_clip_h264.mp4': scorer      superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2  \
 individuals                                                    animal0   
 bodyparts                                                         nose   
 coords                                                               x   
 0                                                   655.882812           
 1                                                   655.882812           
 2                                                   655.882812           
 3                                                   655.882812           
 4                                                   655.882812           
 ..                                                         ...           
 175                                                 896.960938           
 176                                                 772.054688           
 177                                    